# SE4050 Deep Learning Assignment - Human Activity Recognition (HAR)
## Component 1: Transformer Encoder Architecture

**Author:** Dharana  
**Component:** Transformer Encoder for Multi-channel Sensor Time-Series  
**Framework:** TensorFlow / Keras (Trained from Scratch)  

### Team Project Context
In this project, our four-member team compares four deep learning architectures on the UCI Human Activity Recognition Using Smartphones dataset:
* **Dharana (Me):** Transformer encoder (Trained from scratch)
* **Member 2:** 1D-CNN and Exploratory Data Analysis (EDA)
* **Member 3:** BiLSTM and Shared Data Loader
* **Member 4:** CNN-LSTM and Shared Evaluation Pipeline

### Architecture Overview
* **Input:** Batches of sensor windows shaped `(N, 128, 9)` (128 time steps, 9 sensor channels)
* **Linear Projection:** 9 $\to$ 64 dimensions (`d_model = 64`)
* **Trainable Positional Embeddings:** Learnable vectors for temporal position encoding
* **Transformer Encoder Blocks (2 layers):**
  * Multi-Head Attention (`num_heads = 4`, `key_dim = 16`)
  * Pre-LayerNormalization before MHA and FFN
  * Residual skip connections
  * Feed-Forward Network: 128 units (GELU) $\to$ 64 units
  * Dropout: 0.2
* **Global Average Pooling 1D:** Temporal reduction over 128 steps
* **Dense Classification Head:** 64 units (ReLU) $\to$ Dropout(0.2)
* **Output:** 6-unit Softmax probabilities for activity classes (0-5)

### 1. Environment Setup & Imports
Importing reusable modules from `src` to avoid duplicate implementations.

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import tensorflow as tf
import keras

from src.models.transformer import (
    build_transformer_classifier,
    compile_transformer_model,
    load_transformer_model,
    predict_transformer,
    TrainablePositionalEmbedding,
    TransformerEncoderBlock,
)
from src.data_contract import (
    validate_har_dataset,
    generate_synthetic_har_data,
    load_har_npz,
    ACTIVITY_LABEL_MAPPING,
    SENSOR_CHANNEL_NAMES,
)
from src.train_transformer import train_transformer_pipeline, load_config

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")
print(f"GPU Devices:        {tf.config.list_physical_devices('GPU')}")

### 2. Configuration Inspection
Loading the baseline hyperparameter configuration from `configs/transformer.json`.

In [ ]:
config_path = os.path.join(project_root, 'configs', 'transformer.json')
config = load_config(config_path)
print("Loaded Configuration:")
print(json.dumps(config, indent=2))

### 3. Data Contract & Subject Leakage Prevention
We verify the shared contract ensuring:
1. Shapes are `(N, 128, 9)`
2. Labels are integers 0-5
3. **Zero subject overlap** between training and validation splits (`set(subject_train) ∩ set(subject_val) == ∅`)

In [ ]:
# Load real dataset if available, otherwise use synthetic verification data
npz_data_path = os.path.join(project_root, config['data']['npz_path'])
if os.path.exists(npz_data_path):
    print(f"Loading real preprocessed HAR dataset from {npz_data_path}...")
    dataset = load_har_npz(npz_data_path, normalize=config['data'].get('normalize', False))
else:
    print("Real dataset not found; generating synthetic HAR dataset for contract verification...")
    dataset = generate_synthetic_har_data(n_train_windows_per_class=30, n_val_windows_per_class=10, seed=42)

validate_har_dataset(dataset, check_test=True)

X_train, y_train, sub_train = dataset['X_train'], dataset['y_train'], dataset['subject_train']
X_val, y_val, sub_val = dataset['X_val'], dataset['y_val'], dataset['subject_val']

print(f"Training set:   X={X_train.shape}, y={y_train.shape}, Unique Subjects={np.unique(sub_train).tolist()}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}, Unique Subjects={np.unique(sub_val).tolist()}")
print(f"Subject overlap count: {len(set(sub_train).intersection(set(sub_val)))}")

### 4. Build Transformer Model & Inspect Summary

In [ ]:
model = build_transformer_classifier(config=config['model'])
model = compile_transformer_model(model, learning_rate=config['training']['learning_rate'])
model.summary()

### 5. Training Pipeline Execution
Executing the training pipeline with `EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`, and structured output logging.

In [ ]:
# Run training pipeline (using demo settings or full epochs)
best_model, metadata, run_dir = train_transformer_pipeline(
    config=config,
    data=dataset,
    verbose=1
)

### 6. Training History Visualization

In [ ]:
history_path = os.path.join(run_dir, 'history.json')
with open(history_path, 'r') as f:
    history = json.load(f)

epochs_range = range(1, len(history['loss']) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['loss'], label='Train Loss', color='royalblue')
plt.plot(epochs_range, history['val_loss'], label='Val Loss', color='darkorange')
plt.title('Loss Progression')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['accuracy'], label='Train Accuracy', color='royalblue')
plt.plot(epochs_range, history['val_accuracy'], label='Val Accuracy', color='darkorange')
plt.title('Accuracy Progression')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7. Model Serialization & Reload Verification
Confirming that saved `.keras` model reloads properly and inference matches original weights exactly.

In [ ]:
best_model_path = os.path.join(run_dir, 'best_model.keras')
reloaded_model = load_transformer_model(best_model_path)

# Sample evaluation batch
sample_x = dataset.get('X_test', dataset['X_val'])[:10]
sample_y = dataset.get('y_test', dataset['y_val'])[:10]

preds = predict_transformer(reloaded_model, sample_x)
predicted_classes = np.argmax(preds, axis=-1)

print("Sample Predictions vs Ground Truth:")
for i in range(len(sample_x)):
    true_act = ACTIVITY_LABEL_MAPPING.get(int(sample_y[i]), str(sample_y[i]))
    pred_act = ACTIVITY_LABEL_MAPPING.get(int(predicted_classes[i]), str(predicted_classes[i]))
    conf = preds[i, predicted_classes[i]] * 100
    print(f"Sample {i+1:02d} | True: {true_act:<20} | Pred: {pred_act:<20} | Confidence: {conf:5.1f}%")